# 06 Live region demo (on top of the five layers)

A visual run of the whole chain on real, live satellite data for an Indian paddy district: Sentinel-1
radar passes from a public STAC catalogue (Microsoft Planetary Computer, `sentinel-1-rtc`), a dry pass
before the monsoon and a wet pass in July, a cropland mask from ESA WorldCover as the paddy stand-in,
a per-pixel flood probability, pseudo-units, always-answered calls with P_u, a day-route view and
evidence cards with real scene IDs and receipts.

**There is no field ground truth for this region, so this notebook makes no accuracy claim.** Every
number below is a map-only estimate shown for demonstration. In particular:
* the units are a 4 x 4 grid of pseudo-units over a box in the district, not insurance units;
* the flood probability is a logistic of the distance to an Otsu threshold with a scale fixed by hand,
  **not calibrated on this region** (layer 2 calibrates on Sen1Floods11, not here);
* WorldCover cropland (class 40) is used as the paddy mask; it is not a paddy map;
* there are no field checks (n = 0 for every unit), so s_u comes from an assumed map-error floor of 0.05
  and an effective sample of one pixel in 25 (spatial correlation), both stated assumptions;
* the road graph is a stand-in lattice over the box with segments marked flooded from the radar map,
  not OpenStreetMap.

The notebook is **guarded**: with no network (or with `PADDYBIRD_OFFLINE=1`) every live step is
skipped cleanly, a manifest records the skip, and the final check still passes.

In [1]:
%matplotlib inline
import os, json, hashlib, pathlib, platform, datetime as dt
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from paddybird import intake as I, card as C, views as V

ROOT = pathlib.Path.cwd()
if not (ROOT / "paddybird").exists():
    ROOT = ROOT / "notebooks"
OUT = ROOT / "artifacts" / "demo"
OUT.mkdir(parents=True, exist_ok=True)
SEED, TAU = 0, I.TAU
REGION = dict(name="Barpeta district, Assam (box inside the district)", state="Assam",
              bbox=[90.95, 26.30, 91.15, 26.45])        # lon_min, lat_min, lon_max, lat_max (about 20 x 17 km)
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
YEARS = [2026, 2025, 2024]                              # newest first; the first year with both passes is used
DRY = ("03-01", "04-30")                                # before the monsoon
WET = ("07-01", "07-31")                                # Brahmaputra flood season
GRID_DEG = 0.0005                                       # about 50 m read grid
STATUS = {"live": False, "reason": ""}

if os.environ.get("PADDYBIRD_OFFLINE") == "1":
    STATUS["reason"] = "PADDYBIRD_OFFLINE=1"
else:
    try:
        import requests
        requests.get(STAC_URL, timeout=15).raise_for_status()
        import pystac_client, planetary_computer, rasterio   # noqa: F401
        STATUS["live"] = True
    except Exception as exc:
        STATUS["reason"] = f"offline or catalogue unreachable: {type(exc).__name__}"
LIVE = STATUS["live"]
print("live:", LIVE, STATUS["reason"])

live: False PADDYBIRD_OFFLINE=1


## 1 Find the radar passes

Search the catalogue for Sentinel-1 RTC items over the box. The dry pass is the earliest full-cover
item in March to April; the wet pass is the full-cover July item closest to mid-July on the **same
relative orbit**, so both passes see the ground from the same geometry.

In [2]:
def skip(stage, exc):
    global LIVE
    LIVE = False
    STATUS.update(live=False, reason=f"{stage}: {type(exc).__name__}: {str(exc)[:160]}")
    print("skipping the rest:", STATUS["reason"])

scenes, items = [], {}
if LIVE:
    try:
        import pystac_client, planetary_computer
        from shapely.geometry import box, shape
        cat = pystac_client.Client.open(STAC_URL, modifier=planetary_computer.sign_inplace)
        aoi = box(*REGION["bbox"])
        def search(start, end):
            res = cat.search(collections=["sentinel-1-rtc"], bbox=REGION["bbox"], datetime=f"{start}/{end}", max_items=60)
            return [it for it in res.items() if shape(it.geometry).contains(aoi)]
        for y in YEARS:
            dry = sorted(search(f"{y}-{DRY[0]}", f"{y}-{DRY[1]}"), key=lambda it: it.datetime)
            wet = search(f"{y}-{WET[0]}", f"{y}-{WET[1]}")
            mid = dt.datetime(y, 7, 15, tzinfo=dt.timezone.utc)
            pairs = [(d, w) for d in dry for w in wet if d.properties.get("sat:relative_orbit") == w.properties.get("sat:relative_orbit")]
            if pairs:
                d, w = min(pairs, key=lambda p: (abs((p[1].datetime - mid).total_seconds()), p[0].datetime))
                items = {"dry": d, "wet": w}
                break
        if not items:
            raise RuntimeError("no dry and wet pair on one orbit in " + ", ".join(map(str, YEARS)))
        for k, it in items.items():
            scenes.append(dict(id=it.id, date=it.datetime.date().isoformat(), sensor=f"Sentinel-1 RTC VV ({k} pass)",
                               orbit=it.properties.get("sat:relative_orbit"), platform=it.properties.get("platform")))
        display(pd.DataFrame(scenes))
    except Exception as exc:
        skip("scene search", exc)

## 2 Read VV on one grid and map water

Both passes are read on the same 50 m latitude-longitude grid (decimated reads from the cloud-optimised
GeoTIFFs), converted to decibels. Water is dark in VV. The Otsu threshold is found on the wet pass; a
pixel's flood probability is P(water in the wet pass) times P(not water in the dry pass), each a
logistic of the distance to the threshold with a 1 dB scale (uncalibrated, see the top).

In [3]:
def read_on_grid(href, bbox, shape, nearest=False):
    import rasterio
    from rasterio.vrt import WarpedVRT
    from rasterio.enums import Resampling
    from rasterio.transform import from_bounds
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_TIMEOUT="60", GDAL_HTTP_MAX_RETRY="3"):
        with rasterio.open(href) as src:
            t = from_bounds(*bbox, shape[1], shape[0])
            with WarpedVRT(src, crs="EPSG:4326", transform=t, width=shape[1], height=shape[0],
                           resampling=Resampling.nearest if nearest else Resampling.average) as vrt:
                a = vrt.read(1).astype("float32")
                if vrt.nodata is not None:
                    a[a == vrt.nodata] = np.nan
                return a

def otsu(x, bins=256):
    x = x[np.isfinite(x)]
    h, e = np.histogram(x, bins=bins)
    c = (e[:-1] + e[1:]) / 2
    w0 = np.cumsum(h); w1 = w0[-1] - w0
    m0 = np.cumsum(h * c) / np.maximum(w0, 1); m1 = (np.sum(h * c) - np.cumsum(h * c)) / np.maximum(w1, 1)
    return float(c[np.argmax(w0 * w1 * (m0 - m1) ** 2)])

sig = lambda z: 1 / (1 + np.exp(-z))
W_, H_ = int(round((REGION["bbox"][2] - REGION["bbox"][0]) / GRID_DEG)), int(round((REGION["bbox"][3] - REGION["bbox"][1]) / GRID_DEG))
if LIVE:
    try:
        db = {}
        for k, it in items.items():
            vv = read_on_grid(it.assets["vv"].href, REGION["bbox"], (H_, W_))
            vv[vv <= 0] = np.nan
            db[k] = 10 * np.log10(vv)
        T = otsu(np.clip(db["wet"], -30, 5))
        P_flood = sig((T - db["wet"]) / 1.0) * (1 - sig((T - db["dry"]) / 1.0))
        P_flood[~(np.isfinite(db["wet"]) & np.isfinite(db["dry"]))] = np.nan
        print(f"grid {H_} x {W_}, Otsu threshold {T:.2f} dB, valid pixels {np.isfinite(P_flood).mean():.1%}")
    except Exception as exc:
        skip("radar read", exc)

## 3 Paddy stand-in mask

ESA WorldCover 10 m (2021) read on the same grid with nearest-neighbour sampling; class 40
(cropland) is kept. If the mask cannot be read, every land pixel is kept and the manifest says so.

In [4]:
if LIVE:
    try:
        wc = list(cat.search(collections=["esa-worldcover"], bbox=REGION["bbox"], max_items=8).items())
        newest = max(w.datetime.year if w.datetime else int(str(w.properties.get("start_datetime", "0"))[:4]) for w in wc)
        wc = [w for w in wc if (w.datetime.year if w.datetime else int(str(w.properties.get("start_datetime", "0"))[:4])) == newest]
        wc_ids = [w.id for w in wc]
        lc = np.full((H_, W_), np.nan, "float32")
        for w in wc:
            a = read_on_grid(w.assets["map"].href, REGION["bbox"], (H_, W_), nearest=True)
            lc = np.where(np.isnan(lc) & np.isfinite(a) & (a > 0), a, lc)
        paddy = lc == 40
        MASK_SOURCE = f"ESA WorldCover class 40 ({', '.join(wc_ids)})"
        scenes += [dict(id=i, date=str(newest), sensor="ESA WorldCover 10 m (mask)") for i in wc_ids]
    except Exception as exc:
        paddy = np.isfinite(P_flood)
        MASK_SOURCE = f"no mask (WorldCover read failed: {type(exc).__name__}); all valid pixels kept"
    print("paddy mask:", MASK_SOURCE, f"cover {np.nanmean(paddy):.1%}")

In [5]:
if LIVE:
    ext = [REGION["bbox"][0], REGION["bbox"][2], REGION["bbox"][1], REGION["bbox"][3]]
    fig, axs = plt.subplots(1, 4, figsize=(15, 3.6))
    for ax, k in zip(axs[:2], ("dry", "wet")):
        im = ax.imshow(db[k], extent=ext, cmap="gray", vmin=-25, vmax=0)
        ax.set_title(f"VV {k} pass {items[k].datetime.date()} (dB)", fontsize=9)
    fig.colorbar(im, ax=axs[1], shrink=0.7)
    im = axs[2].imshow(P_flood, extent=ext, cmap="Blues", vmin=0, vmax=1); axs[2].set_title("P(newly flooded), uncalibrated", fontsize=9)
    fig.colorbar(im, ax=axs[2], shrink=0.7)
    axs[3].imshow(np.where(paddy, P_flood, np.nan), extent=ext, cmap="Blues", vmin=0, vmax=1); axs[3].set_title("same, on the cropland mask", fontsize=9)
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"{REGION['name']}: live Sentinel-1, no ground truth", fontsize=10)
    fig.tight_layout(); fig.savefig(OUT / "radar_maps.png", dpi=130); plt.show()

## 4 Pseudo-units, calls and P_u

theta_u is the mean flood probability over the unit's cropland pixels (map only, no checks). The call is
always given. s_u^2 = theta_u (1 - theta_u) / n_eff + 0.05^2 with n_eff = (cropland pixels) / 25.

In [6]:
if LIVE:
    try:
        NU = 4
        x0, y0, x1, y1 = REGION["bbox"]
        rows = []
        for j in range(NU):
            for i in range(NU):
                r0, r1 = H_ - (j + 1) * H_ // NU, H_ - j * H_ // NU            # image rows go north to south
                c0, c1 = i * W_ // NU, (i + 1) * W_ // NU
                p, m = P_flood[r0:r1, c0:c1], paddy[r0:r1, c0:c1] & np.isfinite(P_flood[r0:r1, c0:c1])
                n = int(m.sum())
                th = float(p[m].mean()) if n else float(np.nanmean(p))
                s = float(np.sqrt(th * (1 - th) / max(n / 25, 1) + 0.05 ** 2))
                Pu = float(I._phi((th - TAU) / s))
                rows.append(dict(unit_id=f"B{j * NU + i + 1:02d}", name=f"Barpeta pseudo-unit {j * NU + i + 1:02d}",
                                 lon=x0 + (i + 0.5) * (x1 - x0) / NU, lat=y0 + (j + 0.5) * (y1 - y0) / NU,
                                 half=min(x1 - x0, y1 - y0) / NU / 2, theta=th, s=s, call="above" if th > TAU else "below",
                                 P_u=Pu, n_checks=0, cropland_px=n))
        units = pd.DataFrame(rows)
        table = V.unit_calls_table(units, TAU)
        display(HTML(V.table_html(table, 16)))
        fig, ax = plt.subplots(figsize=(6.4, 5))
        V.plot_unit_map(units, ax, tau=TAU, title="Pseudo-unit calls (map only, no ground truth)")
        fig.tight_layout(); fig.savefig(OUT / "unit_map.png", dpi=150); plt.show()
    except Exception as exc:
        skip("unit estimates", exc)

## 5 Day-route view on a stand-in road lattice

A 9 x 9 lattice over the box; a segment is flooded when the mean flood probability along it exceeds
0.4 (8 times slower). Two officers, 4 hour day, cheapest insertion by the chance the call is wrong per
extra minute (the tool's fixture planner; layer 4 owns the real one). Flooded roads in blue.

In [7]:
if LIVE:
    try:
        G = 9
        xs, ys = np.linspace(x0, x1, G), np.linspace(y0, y1, G)
        XX, YY = np.meshgrid(xs, ys)
        nodes = pd.DataFrame({"node": np.arange(G * G), "lon": XX.ravel(), "lat": YY.ravel()})
        def p_at(lon, lat):
            c = int(np.clip((lon - x0) / GRID_DEG, 0, W_ - 1)); r = int(np.clip((y1 - lat) / GRID_DEG, 0, H_ - 1))
            v = P_flood[max(r - 3, 0):r + 4, max(c - 3, 0):c + 4]
            return float(np.nanmean(v)) if np.isfinite(v).any() else 0.0
        edges = []
        for jj in range(G):
            for ii in range(G):
                a = jj * G + ii
                for b in ([a + 1] if ii < G - 1 else []) + ([a + G] if jj < G - 1 else []):
                    pa, pb = nodes.loc[a, ["lon", "lat"]].to_numpy(), nodes.loc[b, ["lon", "lat"]].to_numpy()
                    pf = np.mean([p_at(*(pa + t * (pb - pa))) for t in np.linspace(0, 1, 5)])
                    fl = bool(pf > 0.4)
                    edges.append((a, b, 4.0 * (I.FLOOD_X if fl else 1.0), fl))
        edges = pd.DataFrame(edges, columns=["a", "b", "minutes", "flooded"])
        demo_inp = I.Inputs(units=units, roads_nodes=nodes, roads_edges=edges, routes=[], scenes=scenes,
                            checks=pd.DataFrame(), region=REGION)
        routes = I.plan_routes(demo_inp, n_officers=2, day_minutes=240.0)
        fig, ax = plt.subplots(figsize=(6.6, 5.6))
        V.plot_route_view(nodes, edges, routes, units, ax,
                          title="Stand-in roads (blue = flooded from radar), day routes; no ground truth")
        fig.tight_layout(); fig.savefig(OUT / "route_view.png", dpi=150); plt.show()
        for r in routes:
            print(f"{r['officer']}: {' > '.join(r['stops'])}, {r['minutes']:.0f} min")
    except Exception as exc:
        skip("route view", exc)

## 6 Evidence cards

Cards for the two units whose map-only call is most likely wrong. Scene IDs are the real catalogue
item IDs; the receipt hashes the inputs (including the scene IDs and the stated assumptions) and
outputs.

In [8]:
cards = {}
if LIVE:
    try:
        params = dict(otsu_db=round(T, 3), logistic_scale_db=1.0, map_error_floor=0.05, n_eff_divisor=25,
                      grid_deg=GRID_DEG, mask=MASK_SOURCE, calibrated=False, ground_truth=False)
        for uid in units.unit_id:
            row = units.set_index("unit_id").loc[uid].to_dict(); row["unit_id"] = uid
            cards[uid] = C.build_card(row, scenes, [], tau=TAU, params=params, sources={"stac": STAC_URL, "region": REGION["name"]})
        for uid in table.unit_id[:2]:
            display(HTML(C.card_html(cards[uid])))
    except Exception as exc:
        skip("cards", exc)

## 7 Artifacts and checks

Written to `artifacts/demo/`: `unit_calls.csv`, `cards.json`, figures, and `MANIFEST.json` with the
status (live or skipped), scene IDs, assumptions and the explicit statement that no accuracy claim is
made. The checks are structural only: every pseudo-unit has a call, P_u is in [0, 1], receipts
re-derive and the scene IDs are real catalogue IDs (not the `SYNTH_` fixture). Offline, the check is
that the skip was recorded.

In [9]:
def sha(p):
    return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()[:16]

if LIVE:
    table.to_csv(OUT / "unit_calls.csv", index=False)
    (OUT / "cards.json").write_text(json.dumps(cards, indent=1, default=str))
files = {p.name: sha(p) for p in sorted(OUT.glob("*")) if p.is_file() and p.name != "MANIFEST.json"}
manifest = dict(notebook="06_live_region_demo.ipynb", status="live" if LIVE else "skipped",
                reason=STATUS["reason"], region=REGION, scenes=scenes if LIVE else [],
                ground_truth=False, accuracy_claim=None,
                statement="No field ground truth exists for this region; no accuracy claim is made.",
                assumptions=["pseudo-units (4 x 4 grid), not insurance units", "flood probability not calibrated on this region",
                             "WorldCover cropland as paddy stand-in", "no field checks; s_u from a 0.05 floor and n_eff = px/25",
                             "stand-in road lattice, not OSM"],
                run_utc=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__),
                code_sha256_16={f"paddybird/{n}": sha(ROOT / "paddybird" / n) for n in ("intake.py", "card.py", "views.py")},
                files=files)
(OUT / "MANIFEST.json").write_text(json.dumps(manifest, indent=1, default=str))
print("status:", manifest["status"], "|", manifest["statement"])

status: skipped | No field ground truth exists for this region; no accuracy claim is made.


In [10]:
m = json.loads((OUT / "MANIFEST.json").read_text())
assert m["ground_truth"] is False and m["accuracy_claim"] is None
if m["status"] == "live":
    assert len(units) == 16 and units.call.isin(C.CALLS).all()
    assert units.P_u.between(0, 1).all() and units.P_u.notna().all()
    assert all(C.verify_receipt(c["receipt"]) for c in cards.values())
    assert scenes and not any(s["id"].startswith("SYNTH_") for s in scenes)
    print(f"live demo passed: {len(units)} pseudo-units all answered, {len(cards)} receipts verified, scenes "
          + ", ".join(s["id"] for s in scenes[:2]))
else:
    assert m["reason"], "a skip must say why"
    print("live demo skipped cleanly:", m["reason"])

live demo skipped cleanly: PADDYBIRD_OFFLINE=1
